In [1]:
import os
import pandas as pd
import numpy as np
import torch
import logging
import hashlib
import tempfile
import zipfile
import requests
from numpy import concatenate, full
from tqdm import tqdm
from fnn.microns.__init__ import scan
from fnn.microns.build import network
from fnn.microns.load import params, units, unit_ids


In [ ]:
 #for each sigma in [.01,.05,.10,.20,.30], multiply by 256
    # for image in images (by images we mean frame variable as defined below, should be 90 constant frames, each image has 90 frames)
        # for each noise each time we add noise (implement with another for loop, we need 1000 iterations(at first do 100, 200)), 
            #    make sure in range (0,256)
            # for each scan
                # collect prediction output
                # concat. outputs for different scans
            # concat response outputs, collect shape
            # shape should be (1000, 90, number neurons) (call this response_shape)
        # concat all the response_shape objects together, shape will be (number of images, 1000, 90, num neurons)
        # calc. sum spike count over 90, shape should be (number images, 1000, number neurons)
        
        # calc. mean over 1000, shape should be (number images, number neurons)
        # calc. var over 1000, shape should be (number images, number neurons)
        # plot mean vs var
        # calc. fano factor var over mean
        # plot fano factor dist. as histogram

In [2]:
def visual_prediction(session, scan_idx, stimuli_noise):
    pred_model, table = scan(session, scan_idx, directory = os.path.join(os.getcwd(), "data","microns")) # look at data/microns/scans.csv for numbers that work
    results = pred_model.predict(stimuli = stimuli_noise)
    results = np.array(results) # should be array regardless
    return results

#### noise and stimuli

In [3]:
frames = concatenate([
    full(shape=[90, 144, 256], dtype="uint8", fill_value=128), # 3 seconds of gray
])

In [4]:
mean = 0
sigma = 5
size = (90,144,256)
noise = np.random.normal(loc = mean, scale = sigma, size = size)

In [21]:
print(f"frames shape: {frames.shape}")
print(f"noise shape: {noise.shape}")

frames shape: (90, 144, 256)
noise shape: (90, 144, 256)


In [6]:
new_frames = (frames+noise).astype("uint8")
print(f"new noise: {new_frames.shape}")

new noise: (90, 144, 256)


#### test prediction

In [7]:
stimuli = new_frames 
test = visual_prediction(5,6,stimuli)

In [30]:
test.head()

,0,1,2,3,4,5,6,7,8,9,...,8582,8583,8584,8585,8586,8587,8588,8589,8590,8591
0,0.345906,0.081978,0.679314,0.967388,0.465034,0.691159,0.561997,0.533086,0.464637,0.304514,...,1.045534,0.385014,0.172131,1.571597,0.309345,0.897361,0.223569,1.166553,0.199737,0.868112
1,0.322397,0.050524,0.981810,0.826699,0.451769,0.732992,0.672574,0.652038,0.424963,0.399744,...,0.713890,0.478078,0.156199,2.043463,0.550507,1.109957,0.207346,1.334545,0.179315,0.665529
2,0.241023,0.037177,0.939062,0.709085,0.408413,0.625836,0.681490,0.598160,0.362211,0.367521,...,0.710292,0.506192,0.103343,2.500931,0.596308,1.694331,0.271991,1.272391,0.234741,0.611925
3,0.189183,0.024411,0.823971,0.625400,0.400095,0.629049,0.669547,0.603529,0.319186,0.363671,...,0.689184,0.508164,0.116475,2.189878,0.550182,1.593598,0.248470,1.186808,0.215840,0.616409
4,0.190859,0.024800,0.663530,0.562098,0.376396,0.562864,0.686108,0.570416,0.311047,0.346646,...,0.624596,0.562920,0.142873,1.873178,0.543422,1.686866,0.285859,0.918636,0.291959,0.719146


In [31]:
test.shape

(90, 8592)

Note: we want to concat. test for different scans, ie run visual_prediction() with different scans and ids and concat those

In [8]:
sessions_scans = [[4,7],[5,6],[5,7],[6,2],[6,4],[6,6],[6,7],
                  [7,3],[7,5],[8,5],[9,3],[9,4],[9,6]]

In [38]:
test1 = visual_prediction(4,7,stimuli)
test2 = visual_prediction(5,6,stimuli)

In [ ]:
print(f"test1 shape: {test1.shape}")
print(f"test2 shape: {test2.shape}")

test1 shape: (90, 7493)
test2 shape: (90, 8592)


#### innermost loop code

In [ ]:
# note: this code works for innermost loop 
for pair in sessions_scans:
    try:
        prediction = visual_prediction(pair[0], pair[1], stimuli)
        final_concat = concatenate((final_concat, prediction), axis = 1)
    except NameError:
        final_concat = visual_prediction(pair[0], pair[1], stimuli)

In [11]:
final_concat.shape

(90, 104171)